In [0]:
import os
import pytest
from pyspark.testing.utils import assertSchemaEqual, assertDataFrameEqual
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [0]:
SCHEMA_DB = "training_bs.user_uenishi"
SCHEMA_SAS = "training_bs.user_uenishi_sas"
YYYYMM = os.environ.get("P_BASE_YM", "200504")

CSV_COMMON = "/Volumes/training_bs/common/result"
CSV_USER = f"/Volumes/training_bs/user_uenishi/output/uriage_jisseki_{YYYYMM}"

tables = [row.name for row in spark.catalog.listTables(SCHEMA_DB)]

In [0]:
class TestTableComparison:
    """DB テーブルと SAS テーブルの比較テスト"""

    @pytest.mark.parametrize("table", tables)
    def test_schema_equal(self, table):
        """スキーマのカラム名・データ型が一致すること"""
        sdf_db = spark.read.table(f"{SCHEMA_DB}.{table}")
        sdf_sas = spark.read.table(f"{SCHEMA_SAS}.{table}")
        assertSchemaEqual(sdf_db.schema, sdf_sas.schema)

    @pytest.mark.parametrize("table", tables)
    def test_count_equal(self, table):
        """レコード件数が一致すること"""
        sdf_db = spark.read.table(f"{SCHEMA_DB}.{table}")
        sdf_sas = spark.read.table(f"{SCHEMA_SAS}.{table}")
        assert sdf_db.count() == sdf_sas.count(), \
            f"テーブル {table}: DB={sdf_db.count()}, SAS={sdf_sas.count()}"

    @pytest.mark.parametrize("table", tables)
    def test_data_equal(self, table):
        """全レコードの値が完全一致すること"""
        sdf_db = spark.read.table(f"{SCHEMA_DB}.{table}")
        sdf_sas = spark.read.table(f"{SCHEMA_SAS}.{table}")
        assertDataFrameEqual(sdf_db, sdf_sas)

In [0]:
class TestCsvComparison:
    """CSV 出力の比較テスト"""

    def test_csv_schema_equal(self):
        """CSV スキーマのカラム名・データ型が一致すること"""
        sdf_common = spark.read.csv(CSV_COMMON, header=True, inferSchema=True)
        sdf_user = spark.read.csv(CSV_USER, header=True, inferSchema=True)
        assertSchemaEqual(sdf_common.schema, sdf_user.schema)

    def test_csv_count_equal(self):
        """CSV のレコード件数が一致すること"""
        sdf_common = spark.read.csv(CSV_COMMON, header=True, inferSchema=True)
        sdf_user = spark.read.csv(CSV_USER, header=True, inferSchema=True)
        assert sdf_common.count() == sdf_user.count(), \
            f"CSV: common={sdf_common.count()}, user={sdf_user.count()}"

    def test_csv_data_equal(self):
        """CSV の全レコードの値が完全一致すること"""
        sdf_common = spark.read.csv(CSV_COMMON, header=True, inferSchema=True)
        sdf_user = spark.read.csv(CSV_USER, header=True, inferSchema=True)
        assertDataFrameEqual(sdf_common, sdf_user)